# 01 — Data Preparation & EDA

**Goal:** Load the IBM Telco Churn dataset, perform basic cleaning, explore churn patterns, and save a clean interim checkpoint.

> **Architecture note:** In the original production project, data was pulled monthly from ClickHouse as `abt_lag1_YYYYMM.parquet` snapshots.
> Here we use the IBM Telco public dataset as a structural substitute to demonstrate the same pipeline design.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

DATA_RAW = '../data/raw/telco_churn_raw.csv'
DATA_INTERIM = '../data/interim/df_base_clean.parquet'

print('Libraries loaded.')

## 1. Load & Basic Info

In [ ]:
df = pd.read_csv(DATA_RAW)
print(f'Shape: {df.shape}')
print(f'\nColumn types:')
print(df.dtypes)
df.head(3)

## 2. Cleaning

In [ ]:
# TotalCharges has whitespace-only strings for new customers (tenure=0)
# Replace with 0 and cast to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)

# Binary encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Standardize binary Yes/No columns to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)

# SeniorCitizen is already 0/1

print('Nulls after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nChurn rate: {df["Churn"].mean():.2%}')
print(f'Class imbalance ratio: {(1 - df["Churn"].mean()) / df["Churn"].mean():.1f}:1')

## 3. EDA — Churn Patterns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Churn Patterns by Key Features', fontsize=16, y=1.01)

# 1. Churn distribution
ax = axes[0, 0]
df['Churn'].value_counts().plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='white')
ax.set_title('Target Distribution')
ax.set_xticklabels(['Retained (0)', 'Churned (1)'], rotation=0)
ax.set_ylabel('Count')

# 2. Tenure distribution by churn
ax = axes[0, 1]
df[df['Churn'] == 0]['tenure'].plot(kind='hist', ax=ax, alpha=0.6, label='Retained', bins=30, color='#2ecc71')
df[df['Churn'] == 1]['tenure'].plot(kind='hist', ax=ax, alpha=0.6, label='Churned', bins=30, color='#e74c3c')
ax.set_title('Tenure Distribution by Churn')
ax.set_xlabel('Tenure (months)')
ax.legend()

# 3. Contract type
ax = axes[0, 2]
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
contract_churn.plot(kind='bar', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Churn Rate by Contract Type')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)

# 4. Monthly Charges
ax = axes[1, 0]
df[df['Churn'] == 0]['MonthlyCharges'].plot(kind='hist', ax=ax, alpha=0.6, label='Retained', bins=30, color='#2ecc71')
df[df['Churn'] == 1]['MonthlyCharges'].plot(kind='hist', ax=ax, alpha=0.6, label='Churned', bins=30, color='#e74c3c')
ax.set_title('Monthly Charges by Churn')
ax.set_xlabel('Monthly Charges ($)')
ax.legend()

# 5. Internet service
ax = axes[1, 1]
inet_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
inet_churn.plot(kind='bar', ax=ax, color='#9b59b6', edgecolor='white')
ax.set_title('Churn Rate by Internet Service')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)

# 6. Payment method
ax = axes[1, 2]
pay_churn = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)
pay_churn.plot(kind='bar', ax=ax, color='#e67e22', edgecolor='white')
ax.set_title('Churn Rate by Payment Method')
ax.set_ylabel('Churn Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20)

plt.tight_layout()
plt.savefig('../reports/01_eda_churn_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

In [ ]:
# Key business insights from EDA
print('=== KEY EDA FINDINGS ===')
print(f"Month-to-month churn rate: {df[df['Contract']=='Month-to-month']['Churn'].mean():.2%}")
print(f"Two-year contract churn rate: {df[df['Contract']=='Two year']['Churn'].mean():.2%}")
print(f"\nFiber optic churn rate: {df[df['InternetService']=='Fiber optic']['Churn'].mean():.2%}")
print(f"DSL churn rate: {df[df['InternetService']=='DSL']['Churn'].mean():.2%}")
print(f"\nMedian tenure (churned): {df[df['Churn']==1]['tenure'].median():.0f} months")
print(f"Median tenure (retained): {df[df['Churn']==0]['tenure'].median():.0f} months")

## 4. Save Interim Checkpoint

In [ ]:
df.to_parquet(DATA_INTERIM, index=False)
print(f'Saved clean base: {df.shape}')
print('\nFinished 01_data_prep_eda.ipynb')